# Quality Assessment

## Imports and Data Reads

In [1]:
import pandas as pd

In [2]:
# orders_cl.csv
url = "https://drive.google.com/file/d/1P3iIvUiG_r2dnfEYHzFxyMrgKbLf1dh_/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders_df = pd.read_csv(path)

# orderlines_cl.csv
url = "https://drive.google.com/file/d/1PgWZa-eeLfQWVviyqL63DDNnUVxO6k50/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines_df = pd.read_csv(path)

# products_cl.csv
url = "https://drive.google.com/file/d/1DUN5FiX2XIWSLCce2cF2fRpLUb2ocSp8/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products_df = pd.read_csv(path)

## Quality exclusions

In [3]:
# exclude orders with items that don't appear in products
orderlines_products = orderlines_df.merge(products_df, how="left", on = "sku")[["order_id","sku","name" ]]

orders_to_delete = orderlines_products.loc[orderlines_products["name"].isna(), "order_id"].unique()

orderlines_df = orderlines_df.loc[~orderlines_df["order_id"].isin(orders_to_delete)]

In [4]:
# keep only completed orders
orders_df = orders_df.loc[orders_df["state"] == "Completed"]

In [5]:
# keep only orders that appear in both orderlines and orders

# order_ids that are in both tables
order_ids_to_keep = (
    orders_df
    .merge(orderlines_df, how="inner", on="order_id")
    ["order_id"].unique()
)

# keep those ids in orders
orders_df = orders_df.loc[orders_df["order_id"].isin(order_ids_to_keep)].copy()

# keep those ids in orderlines
orderlines_df = orderlines_df.loc[orderlines_df["order_id"].isin(order_ids_to_keep)].copy()

In [6]:
# keep only orders with low disagreement between total prices as calculated from orderlines and total_paid from orders

# find difference between two "total" values
orderlines_order_ids = (
    orderlines_df
    .assign(unit_price_total=orderlines_df["product_quantity"]*orderlines_df["unit_price"])
    .groupby("order_id", as_index=False)
    ["unit_price_total"].sum()
)
diff_df = orders_df.merge(orderlines_order_ids, on="order_id")
diff_df["difference"] = diff_df["total_paid"] - diff_df["unit_price_total"]

# calculate the quartiles
Q1 = diff_df["difference"].quantile(0.25)
Q3 = diff_df["difference"].quantile(0.75)

# calculate the interquartile range
IQR = Q3-Q1

# filter to include only "non-outliers" via 1.5*IQR rule
low_mask = diff_df["difference"] >= (Q1 - 1.5*IQR)
high_mask = diff_df["difference"] <= (Q3 + 1.5*IQR)
normal_diff_ids = diff_df.loc[low_mask & high_mask, "order_id"].unique()

orders_df = orders_df.loc[orders_df["order_id"].isin(normal_diff_ids)].copy()
orderlines_df = orderlines_df.loc[orderlines_df["order_id"].isin(normal_diff_ids)].copy()

In [ ]:
# working locally
orders_df.to_csv("../Data/Quality/orders_qu.csv", index=False)
orderlines_df.to_csv("../Data/Quality/orderlines_qu.csv", index=False)